# 🎨 ComfyUI Studio - Google Colab Runtime

This notebook turns your Google Colab instance into a high-performance GPU runtime for **ComfyUI Studio**.

- Persistent models & custom nodes on Google Drive (`MyDrive/ComfyStudio`)
- Secure tunnel exposure (Cloudflared / ngrok / localtunnel)
- Direct connection to your local ComfyUI Studio workstation


In [ ]:
#@title 1. Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/ComfyStudio/models', exist_ok=True)
print("✅ Google Drive mounted at /content/drive/MyDrive/ComfyStudio")

In [ ]:
#@title 2. Setup ComfyUI & Dependencies
!git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements.txt

# Link extra model paths to Google Drive
extra_paths_config = """
colab_drive:
    base_path: /content/drive/MyDrive/ComfyStudio/models
    checkpoints: checkpoint/
    loras: lora/
    vae: vae/
    controlnet: controlnet/
    upscale_models: upscaler/
"""
with open('/content/ComfyUI/extra_model_paths.yaml', 'w') as f:
    f.write(extra_paths_config.strip())

# Install ComfyUI-Manager
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager

print("✅ ComfyUI and models configured!")

In [ ]:
#@title 3. Start ComfyUI and Expose via Cloudflare Tunnel
# Download cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess
import threading
import time
import re

# Start ComfyUI in background
def run_comfy():
    !python main.py --listen 127.0.0.1 --port 8188 --preview-method auto

t = threading.Thread(target=run_comfy)
t.daemon = True
t.start()

time.sleep(3)

# Expose tunnel
p = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print("Waiting for Cloudflare Tunnel URL...")
for line in p.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print("\n=======================================================")
        print(f"🎉 YOUR COMFYUI STUDIO ENDPOINT: {url}")
        print("👉 Copy this URL into ComfyUI Studio -> Runtime / Settings")
        print("=======================================================\n")
        break
